# 1 - Simulación Consolidada y Generación de Datos Sintéticos (FinanceAI)

**1. Simulación** > 2. EDA > 3. Entrenamiento

Este cuaderno genera un conjunto de datos sintéticos autónomo, limpio y **estructuralmente coherente**. En lugar de asignar valores aleatorios ciegos, el perfil financiero de cada usuario (endeudamiento y ahorro) se calcula matemáticamente evaluando su historial anual de transacciones generadas. Esto garantiza una correlación lógica estricta para el entrenamiento de los algoritmos de Machine Learning, replicando el funcionamiento de un entorno de negocio real.

In [1]:
import pandas as pd
import numpy as np
import os
import json

# Fijar semilla de aleatoriedad para garantizar reproducibilidad total en el proyecto
SEED = 42
np.random.seed(SEED)

## 1. Generación de Población Base
Iniciamos creando el núcleo poblacional (1800 usuarios) únicamente con sus ingresos y datos de identidad base.

In [2]:
n_usuarios = 1800

nombres = ['Ana', 'Juan', 'Sofia', 'Pedro', 'Laura', 'Diego', 'Valentina', 'Carlos', 'Camila', 'Luis',
           'Maria', 'Jorge', 'Lucia', 'Miguel', 'Marta', 'Alejandro', 'Elena', 'Martin', 'Clara', 'Andres']

# Generación de ingresos mensuales (sueldos entre 500 y 6000)
ingresos = np.round(np.random.uniform(500, 6000, size=n_usuarios), 2)

df_usuarios = pd.DataFrame({
    'id': range(1, n_usuarios + 1),
    'nombre': np.random.choice(nombres, size=n_usuarios),
    'ingreso_mensual': ingresos
})

print("Estructura base de usuarios generada:", df_usuarios.shape)

Estructura base de usuarios generada: (1800, 3)


## 2. Generación de Transacciones (Big Data)
Generamos 240,000 consumos asociados a los usuarios y limitados estrictamente a las 10 categorías definitivas.

In [3]:
n_transacciones = 240000

# Las 10 categorías oficiales
diccionario_conceptos = {
    'Alimentacion': ['supermercado coto', 'verduleria el sol', 'carniceria central', 'almacen san martin', 'compra panaderia', 'supermercado carrefour', 'compras fiambreria', 'compra dia'],
    'Educacion': ['cuota universidad', 'compra libros', 'curso de programacion', 'matricula colegio', 'utiles escolares', 'taller ingles', 'cuota jardin'],
    'Electrodomesticos': ['compra heladera', 'lavarropas fravega', 'televisor garbarino', 'microondas musimundo', 'licuadora philips', 'pava electrica', 'aire acondicionado'],
    'Inversion': ['compra dolares', 'fondo comun inversion', 'plazo fijo', 'cedears', 'bonos del estado', 'acciones ypf', 'transferencia broker'],
    'Ocio': ['salida cine', 'suscripcion netflix', 'cena restaurante', 'entradas recital', 'cerveceria', 'suscripcion spotify', 'juegos steam', 'cafeteria'],
    'Salud': ['estudios clinicos', 'compra farmacia', 'consulta medica', 'cuota prepaga', 'medicamentos', 'dentista', 'analisis sangre', 'optica'],
    'Servicios': ['factura luz edesur', 'abono internet', 'servicio agua', 'factura gas', 'telefonia movil', 'impuesto municipal', 'abl', 'rentas'],
    'Transporte': ['viaje uber', 'colectivo', 'carga sube', 'combustible ypf', 'peaje autopista', 'taxi', 'viaje cabify', 'combustible shell'],
    'Vestimenta': ['compra zapatillas', 'pantalon jean', 'remera algodon', 'campera invierno', 'ropa deportiva', 'local indumentaria', 'zapatos'],
    'Vivienda': ['pago alquiler', 'expensas edificio', 'servicio plomeria', 'ferreteria', 'pintura habitacion', 'reparacion electrica', 'materiales construccion']
}

# Limites de montos realistas y calibrados en dólares
rangos_montos = {
    'Alimentacion': (5, 100), 'Educacion': (30, 400), 'Electrodomesticos': (200, 1500), 'Inversion': (100, 1500),
    'Ocio': (5, 150), 'Salud': (40, 500), 'Servicios': (20, 150), 'Transporte': (1, 200),
    'Vestimenta': (5, 300), 'Vivienda': (200, 1200)
}

categorias = list(diccionario_conceptos.keys())
usuario_ids = np.random.randint(1, n_usuarios + 1, size=n_transacciones)

# Asignamos categorias simulando comportamientos de gasto frecuentes
prob_categorias = [0.15, 0.05, 0.05, 0.10, 0.10, 0.05, 0.20, 0.10, 0.05, 0.15]
selected_categories = np.random.choice(categorias, size=n_transacciones, p=prob_categorias)

fechas_random = pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 365, size=n_transacciones), unit='d')

descripciones = []
montos = []

for cat in selected_categories:
    desc = np.random.choice(diccionario_conceptos[cat])
    
    # Inyección de ruido en texto (simulando errores de usuario o teclado)
    if np.random.rand() < 0.20:
        prefijos = ["pago ", "compra ", "tarjeta ", "fac ", ""]
        desc = str(np.random.choice(prefijos)) + desc
    if np.random.rand() < 0.10:
        desc = desc.replace("a", "q", 1) if "a" in desc else desc.replace("e", "w", 1)

    min_m, max_m = rangos_montos[cat]
    monto = round(np.random.uniform(min_m, max_m), 2)
    descripciones.append(desc)
    montos.append(monto)

# Ruido de etiquetado en categorías (10%) para evitar que el algoritmo NLP sea 100% perfecto
ruido_mask = np.random.rand(n_transacciones) < 0.10
categorias_ruido = np.random.choice(categorias, size=ruido_mask.sum())
selected_categories[ruido_mask] = categorias_ruido

df_transacciones = pd.DataFrame({
    'id': range(1, n_transacciones + 1),
    'usuario_id': usuario_ids,
    'descripcion': descripciones,
    'valor': montos,
    'categoria': selected_categories,
    'fecha': fechas_random.strftime('%Y-%m-%d')
})

print("Transacciones simuladas:", df_transacciones.shape)

Transacciones simuladas: (240000, 6)


## 3. Consolidación de Perfil Financiero (Cálculo Relacional Coherente)
Derivaremos las variables de endeudamiento y ahorro auditando los consumos de cada usuario. Gracias a la alta densidad de datos (Big Data), el análisis de gastos anuales divididos entre 12 refleja promedios mensuales reales y precisos.

In [4]:
user_profiles = []

for user_id in df_usuarios['id']:
    tx_user = df_transacciones[df_transacciones['usuario_id'] == user_id]
    ingreso = df_usuarios.loc[df_usuarios['id'] == user_id, 'ingreso_mensual'].values[0]
    
    # --- Cálculo de Nivel de Endeudamiento ---
    # Los gastos fijos suman las categorías Vivienda y Servicios a lo largo de 1 año
    gastos_fijos_anual = tx_user[tx_user['categoria'].isin(['Vivienda', 'Servicios'])]['valor'].sum()
    gastos_fijos_mensuales = gastos_fijos_anual / 12.0
    
    endeudamiento = round((gastos_fijos_mensuales / ingreso) * 100, 2)
    endeudamiento = max(5.0, min(endeudamiento, 90.0)) # Clampeamos entre limites racionales
    
    # --- Cálculo de Frecuencia de Ahorro ---
    # Segun transacciones de Inversion mensuales a lo largo del año
    n_inversion_anual = len(tx_user[tx_user['categoria'] == 'Inversion'])
    inversiones_mensuales = n_inversion_anual / 12.0
    
    if inversiones_mensuales == 0:
        frecuencia_ahorro = 'Ninguna'
    elif inversiones_mensuales < 1.0:
        frecuencia_ahorro = 'Baja'
    elif inversiones_mensuales < 3.0:
        frecuencia_ahorro = 'Media'
    else:
        frecuencia_ahorro = 'Alta'
        
    # --- Resolución del Perfil (Clasificación de Salud) ---
    if endeudamiento > 50.0:
        perfil = 'En riesgo'
    elif endeudamiento > 35.0 or frecuencia_ahorro == 'Ninguna':
        perfil = 'En observacion'
    elif endeudamiento <= 35.0 and frecuencia_ahorro in ['Media', 'Alta']:
        perfil = 'Saludable'
    else:
        perfil = 'En observacion'
        
    # Inyección de 15% de ruido para retar al algoritmo
    if np.random.rand() < 0.15:
        perfil = np.random.choice(['Saludable', 'En observacion', 'En riesgo'])
        
    user_profiles.append({
        'id': user_id,
        'nivel_endeudamiento': endeudamiento,
        'frecuencia_ahorro': frecuencia_ahorro,
        'perfil_financiero': perfil
    })

df_calc = pd.DataFrame(user_profiles)
df_usuarios = pd.merge(df_usuarios, df_calc, on='id')

print("Perfiles consistentes calculados. Distribución:")
print(df_usuarios['perfil_financiero'].value_counts(normalize=True) * 100)

Perfiles consistentes calculados. Distribución:
perfil_financiero
En riesgo         38.444444
En observacion    33.222222
Saludable         28.333333
Name: proportion, dtype: float64


## 4. Prevención de Data Leakage (Dual Split MLOps)
Aplicamos el paradigma dual para prevención total de fugas:
*   **Para Perfiles (Usuarios):** Aplicamos un corte *Cross-Sectional* tradicional.
*   **Para Transacciones (NLP):** Aplicamos un corte *Out-of-Time* (temporal). El modelo entrena con el historial de Ene-Ago y se evalúa de manera ciega sobre los gastos del futuro (Nov-Dic).

In [5]:
# 1. Split Transversal (Group K-Fold) para Usuarios
splits_usr = np.random.choice(['train', 'val', 'test'], size=n_usuarios, p=[0.6, 0.2, 0.2])
df_usuarios['split'] = splits_usr

# 2. Split Temporal (Out-of-Time) para Transacciones
# Enero a Agosto (train), Sept-Oct (val), Nov-Dic (test)
meses = pd.to_datetime(df_transacciones['fecha']).dt.month
condiciones = [
    meses <= 8,
    meses.isin([9, 10]),
    meses >= 11
]
df_transacciones['split'] = np.select(condiciones, ['train', 'val', 'test'])

print("Distribución Transversal de Usuarios (Cross-sectional):")
print(df_usuarios['split'].value_counts(normalize=True) * 100)
print("\nDistribución Temporal de Transacciones (Out-of-Time):")
print(df_transacciones['split'].value_counts(normalize=True) * 100)

Distribución Transversal de Usuarios (Cross-sectional):
split
train    61.555556
val      19.777778
test     18.666667
Name: proportion, dtype: float64

Distribución Temporal de Transacciones (Out-of-Time):
split
train    66.457917
val      16.794583
test     16.747500
Name: proportion, dtype: float64


## 5. Exportación de Archivos Semilla
Exportamos tablas robustas y finalizadas a `data/`.

In [6]:
os.makedirs('data', exist_ok=True)

df_usuarios.to_csv('data/usuarios.csv', index=False)
df_transacciones.to_csv('data/transacciones.csv', index=False)

df_usuarios.to_json('data/usuarios.json', orient='records', indent=2, force_ascii=False)
df_transacciones.to_json('data/transacciones.json', orient='records', indent=2, force_ascii=False)

print("¡Exportación exitosa! Semillas listas para el pipeline.")

¡Exportación exitosa! Semillas listas para el pipeline.
